## Auto Loader — CSV to Bronze Delta Table
Reads any new CSV file (with header) landing in `/Volumes/retail/blob_files/volumes/` and appends it to the Delta table `retail.blob_bronze.transactions`.

In [0]:
source_path         = "/Volumes/retail/blob_files/volumes/"
schema_location     = "/Volumes/retail/blob_files/volumes/_schema"
checkpoint_location = "/Volumes/retail/blob_files/volumes/_checkpoint"
target_table        = "retail.blob_bronze.transactions"

# Read: detect every new CSV file that lands in the source path
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)  # persists inferred schema across runs
    .option("cloudFiles.inferColumnTypes", "true")          # infer proper column types
    .option("header", "true")                               # first row = column names
    .load(source_path)
)

# Write: append new records to the Bronze Delta table
# availableNow=True — processes all pending files as a batch, then stops
query = (
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()
print(f"Load complete. Data written to {target_table}")

In [0]:
%sql
SELECT COUNT(*) AS total_rows FROM retail.blob_bronze.transactions